In [56]:
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml import Pipeline, PipelineModel

from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [13]:
import os
from pyspark.sql import SparkSession

# Инициализируем локальную сессию Spark, используя все доступные ядра процессора [*]
spark = SparkSession.builder \
    .appName("PySpark_ML_Demo") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Session успешно создана! Версия Spark: {spark.version}")

Spark Session успешно создана! Версия Spark: 3.5.0


In [4]:
!ls

160.94s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
work


In [5]:
(df := spark.read.csv("work/data/creditcard.csv", header=True, inferSchema=True))

DataFrame[Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, V25: double, V26: double, V27: double, V28: double, Amount: double, Class: int]

In [6]:
df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [7]:
df.show(5)

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|                V1|                 V2|              V3|                V4|                 V5|                 V6|                 V7|                V8|                V9|                V10|               V11|               V12|               V13|               V14|               V15|               V16|               V17|                V18|               V19|                V20|                 V21|                V22|     

# EDA

In [8]:
print(f"Количество строк: {df.count()}")
print(f"Количество столбцов: {len(df.columns)}")

Количество строк: 284807
Количество столбцов: 31


In [9]:
df.describe().show()

+-------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+
|summary|             Time|                  V1|                  V2|                  V3|                  V4|                  V5|                  V6|                  V7|                  V8|                  V9|                 V10|                 V11|                 V12|                 V13|                 V14|                 V15|  

In [10]:
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|   0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|    0|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [11]:
class_counts = df.groupBy("Class").count()
class_counts.show()

+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [12]:
total_count = df.count()
fraud_count = class_counts.filter(F.col("Class") == 1).collect()[0]["count"]
normal_count = class_counts.filter(F.col("Class") == 0).collect()[0]["count"]

print(f"Нормальных транзакций (0): {normal_count} ({normal_count/total_count*100:.2f}%)")
print(f"Мошеннических транзакций (1): {fraud_count} ({fraud_count/total_count*100:.2f}%)")

Нормальных транзакций (0): 284315 (99.83%)
Мошеннических транзакций (1): 492 (0.17%)


Классы сильно дисбалансированы, поэтому нужны будут метрики, сильно штрафующие за неправильное предсказание малого класса.

# Пайплайн

## train/test

Классы дисбалансированы, при разделении на train/test нужно делать стратификацию.

In [14]:
df_0 = df.filter(F.col("Class") == 0)
df_1 = df.filter(F.col("Class") == 1)

In [15]:
train_0, test_0 = df_0.randomSplit([0.8, 0.2], seed=42)
train_1, test_1 = df_1.randomSplit([0.8, 0.2], seed=42)

In [16]:
train_df = train_0.union(train_1)
test_df = test_0.union(test_1)

In [17]:
train_df = train_df.orderBy(F.rand())
test_df = test_df.orderBy(F.rand())

In [18]:
print(f"Train size: {train_df.count()}")
print(f"Test size: {test_df.count()}")

Train size: 228164
Test size: 56643


In [19]:
def class_distribution(df, name):
    total = df.count()
    print(f"Распределение классов в {name} (всего: {total}):")
    df.groupBy("Class") \
      .agg(
          F.count("*").alias("count"),
          F.round(F.count("*") / total * 100, 2).alias("percent")
      ) \
      .orderBy(F.desc("count")) \
      .show()

class_distribution(train_df, "TRAIN")
class_distribution(test_df, "TEST")

Распределение классов в TRAIN (всего: 228164):
+-----+------+-------+
|Class| count|percent|
+-----+------+-------+
|    0|227770|  99.83|
|    1|   394|   0.17|
+-----+------+-------+

Распределение классов в TEST (всего: 56643):
+-----+-----+-------+
|Class|count|percent|
+-----+-----+-------+
|    0|56545|  99.83|
|    1|   98|   0.17|
+-----+-----+-------+



## pipeline

In [20]:
feature_cols = [c for c in df.columns if c != 'Class']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="rawFeatures")
scaler = StandardScaler(inputCol="rawFeatures", outputCol="features", withStd=True, withMean=False)

In [21]:
lr = LogisticRegression(featuresCol="features", labelCol="Class", maxIter=10)

In [22]:
pipeline = Pipeline(stages=[assembler, scaler, lr])

In [23]:
pipeline_model = pipeline.fit(train_df)

In [24]:
predictions = pipeline_model.transform(test_df)

In [25]:
predictions.select("Class", "prediction", "probability").show(10, truncate=False)

+-----+----------+------------------------------------------+
|Class|prediction|probability                               |
+-----+----------+------------------------------------------+
|0    |0.0       |[0.9997162926586485,2.8370734135152453E-4]|
|0    |0.0       |[0.9993542515986095,6.457484013905335E-4] |
|0    |0.0       |[0.9997712330103085,2.287669896915423E-4] |
|0    |0.0       |[0.999892037244263,1.0796275573698999E-4] |
|0    |0.0       |[0.9996867817326266,3.132182673734052E-4] |
|0    |0.0       |[0.9997288206163908,2.7117938360921023E-4]|
|0    |0.0       |[0.9997999532019985,2.0004679800145198E-4]|
|0    |0.0       |[0.9998393461041704,1.606538958296433E-4] |
|0    |0.0       |[0.9997924913699238,2.0750863007623632E-4]|
|0    |0.0       |[0.9976784006225947,0.0023215993774052812]|
+-----+----------+------------------------------------------+
only showing top 10 rows



In [26]:
binary_evaluator = BinaryClassificationEvaluator(labelCol="Class", rawPredictionCol="rawPrediction")
au_roc = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderROC"})
print(f"Area Under ROC (AUROC): {au_roc:.4f}")

au_pr = binary_evaluator.evaluate(predictions, {binary_evaluator.metricName: "areaUnderPR"})
print(f"Area Under PR (AUPRC): {au_pr:.4f}")


tp = predictions.filter((F.col("Class") == 1) & (F.col("prediction") == 1)).count()
fp = predictions.filter((F.col("Class") == 0) & (F.col("prediction") == 1)).count()
fn = predictions.filter((F.col("Class") == 1) & (F.col("prediction") == 0)).count()
tn = predictions.filter((F.col("Class") == 0) & (F.col("prediction") == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

print(f"\nPrecision: {precision:.4f} (Точность)")
print(f"Recall:    {recall:.4f} (Полнота)")
print(f"F1-score:  {f1:.4f}")
print(f"Accuracy:  {accuracy:.4f}\n")


conf_matrix = predictions.groupBy("Class", "prediction").count() \
    .withColumnRenamed("Class", "Actual") \
    .withColumnRenamed("prediction", "Predicted")

conf_matrix_pivot = conf_matrix.groupBy("Actual").pivot("Predicted").sum("count").fillna(0)
conf_matrix_pivot.show()

Area Under ROC (AUROC): 0.9810
Area Under PR (AUPRC): 0.7560

Precision: 0.8857 (Точность)
Recall:    0.6327 (Полнота)
F1-score:  0.7381
Accuracy:  0.9992

+------+-----+---+
|Actual|  0.0|1.0|
+------+-----+---+
|     1|   36| 62|
|     0|56537|  8|
+------+-----+---+



In [27]:
multi_evaluator = MulticlassClassificationEvaluator(
    labelCol="Class",
    predictionCol="prediction"
)

precision_label = multi_evaluator.evaluate(
    predictions, {multi_evaluator.metricName: "precisionByLabel", multi_evaluator.metricLabel: 1.0}
)
recall_label = multi_evaluator.evaluate(
    predictions, {multi_evaluator.metricName: "recallByLabel", multi_evaluator.metricLabel: 1.0}
)
f1_label = multi_evaluator.evaluate(
    predictions, {multi_evaluator.metricName: "fMeasureByLabel", multi_evaluator.metricLabel: 1.0}
)

In [28]:
precision_label, recall_label, f1_label

(0.8857142857142857, 0.6326530612244898, 0.7380952380952381)

## кросс-валидация

In [29]:
param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .addGrid(lr.maxIter, [50, 100]) \
    .build()

In [30]:
evaluator = BinaryClassificationEvaluator(
    labelCol="Class", 
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

In [31]:
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42
)


In [32]:
cv_model = cv.fit(train_df)

In [33]:
best_model = cv_model.bestModel

In [34]:
lr_best_model = best_model.stages[-1]

print("Лучшие параметры модели")
print(f"regParam (регуляризация): {lr_best_model.getRegParam()}")
print(f"elasticNetParam (L1/L2): {lr_best_model.getElasticNetParam()}")
print(f"maxIter (итерации): {lr_best_model.getMaxIter()}")

Лучшие параметры модели
regParam (регуляризация): 0.01
elasticNetParam (L1/L2): 0.0
maxIter (итерации): 50


In [35]:
predictions_cv = best_model.transform(test_df)

In [36]:
binary_evaluator = BinaryClassificationEvaluator(labelCol="Class", rawPredictionCol="rawPrediction")
au_roc = binary_evaluator.evaluate(predictions_cv, {binary_evaluator.metricName: "areaUnderROC"})
print(f"Area Under ROC (AUROC): {au_roc:.4f}")

au_pr = binary_evaluator.evaluate(predictions_cv, {binary_evaluator.metricName: "areaUnderPR"})
print(f"Area Under PR (AUPRC): {au_pr:.4f}")


tp = predictions_cv.filter((F.col("Class") == 1) & (F.col("prediction") == 1)).count()
fp = predictions_cv.filter((F.col("Class") == 0) & (F.col("prediction") == 1)).count()
fn = predictions_cv.filter((F.col("Class") == 1) & (F.col("prediction") == 0)).count()
tn = predictions_cv.filter((F.col("Class") == 0) & (F.col("prediction") == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

print(f"\nPrecision: {precision:.4f} (Точность)")
print(f"Recall:    {recall:.4f} (Полнота)")
print(f"F1-score:  {f1:.4f}")
print(f"Accuracy:  {accuracy:.4f}\n")


conf_matrix = predictions_cv.groupBy("Class", "prediction").count() \
    .withColumnRenamed("Class", "Actual") \
    .withColumnRenamed("prediction", "Predicted")

conf_matrix_pivot = conf_matrix.groupBy("Actual").pivot("Predicted").sum("count").fillna(0)
conf_matrix_pivot.show()

Area Under ROC (AUROC): 0.9754
Area Under PR (AUPRC): 0.7522

Precision: 0.8824 (Точность)
Recall:    0.4592 (Полнота)
F1-score:  0.6040
Accuracy:  0.9990

+------+-----+---+
|Actual|  0.0|1.0|
+------+-----+---+
|     1|   53| 45|
|     0|56539|  6|
+------+-----+---+



## Разные методы классификации

In [37]:
def evaluate_model(name, preds):
    print(f"\n{'='*20} Модель: {name} {'='*20}")
    evaluator = BinaryClassificationEvaluator(labelCol="Class", rawPredictionCol="rawPrediction")
    
    au_roc = evaluator.evaluate(preds, {evaluator.metricName: "areaUnderROC"})
    au_pr = evaluator.evaluate(preds, {evaluator.metricName: "areaUnderPR"})
    print(f"AUROC: {au_roc:.4f}")
    print(f"AUPRC: {au_pr:.4f}")
    
    tp = preds.filter((F.col("Class") == 1) & (F.col("prediction") == 1)).count()
    fp = preds.filter((F.col("Class") == 0) & (F.col("prediction") == 1)).count()
    fn = preds.filter((F.col("Class") == 1) & (F.col("prediction") == 0)).count()
    tn = preds.filter((F.col("Class") == 0) & (F.col("prediction") == 0)).count()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}\n")
    
    preds.groupBy("Class", "prediction").count() \
        .withColumnRenamed("Class", "Actual") \
        .withColumnRenamed("prediction", "Predicted") \
        .groupBy("Actual").pivot("Predicted").sum("count").fillna(0).show()

In [38]:
feature_cols = [c for c in df.columns if c != 'Class']
assembler_trees = VectorAssembler(inputCols=feature_cols, outputCol="features")

In [39]:
rf = RandomForestClassifier(labelCol="Class", featuresCol="features", numTrees=50, seed=42)
pipeline_rf = Pipeline(stages=[assembler_trees, rf])

rf_model = pipeline_rf.fit(train_df)
preds_rf = rf_model.transform(test_df)
evaluate_model("Random Forest", preds_rf)


==================== Модель: Random Forest ====================
AUROC: 0.9762
AUPRC: 0.8357
Precision: 0.8889
Recall:    0.7347
F1-score:  0.8045

+------+-----+---+
|Actual|  0.0|1.0|
+------+-----+---+
|     1|   26| 72|
|     0|56536|  9|
+------+-----+---+



In [40]:
gbt = GBTClassifier(labelCol="Class", featuresCol="features", maxIter=50, seed=42)
pipeline_gbt = Pipeline(stages=[assembler_trees, gbt])

gbt_model = pipeline_gbt.fit(train_df)
preds_gbt = gbt_model.transform(test_df)
evaluate_model("GBT", preds_gbt)


==================== Модель: GBT ====================
AUROC: 0.9833
AUPRC: 0.7826
Precision: 0.8539
Recall:    0.7755
F1-score:  0.8128

+------+-----+---+
|Actual|  0.0|1.0|
+------+-----+---+
|     1|   22| 76|
|     0|56532| 13|
+------+-----+---+



## Сохранение на диск

In [45]:
model_path_lr = "work/models/fraud_detection_lr_pipeline"
model_path_rf = "work/models/fraud_detection_rf_pipeline"
model_path_gbt = "work/models/fraud_detection_gbt_pipeline"
os.makedirs(model_path_lr, exist_ok=True)
os.makedirs(model_path_rf, exist_ok=True)
os.makedirs(model_path_gbt, exist_ok=True)

In [49]:
pipeline_model.write().overwrite().save(model_path_lr)
rf_model.write().overwrite().save(model_path_rf)
gbt_model.write().overwrite().save(model_path_gbt)

In [50]:
loaded_model_gbt = PipelineModel.load(model_path_gbt)

In [52]:
predictions_loaded_gbt = loaded_model_gbt.transform(test_df)

In [53]:
predictions_loaded_gbt.select("Class", "prediction", "probability").show(5, truncate=False)

+-----+----------+-----------------------------------------+
|Class|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0    |0.0       |[0.9780377560341623,0.02196224396583768] |
|0    |0.0       |[0.9784606579790809,0.021539342020919117]|
|0    |0.0       |[0.9784890926852263,0.021510907314773675]|
|0    |0.0       |[0.9784900244611217,0.021509975538878345]|
|0    |0.0       |[0.9784416904539496,0.021558309546050425]|
+-----+----------+-----------------------------------------+
only showing top 5 rows



In [54]:
evaluator = BinaryClassificationEvaluator(labelCol="Class", rawPredictionCol="rawPrediction")
original_pr = evaluator.evaluate(preds_gbt, {evaluator.metricName: "areaUnderPR"})
loaded_pr = evaluator.evaluate(predictions_loaded_gbt, {evaluator.metricName: "areaUnderPR"})


In [55]:
print(f"AUPRC оригинальной модели: {original_pr:.4f}")
print(f"AUPRC загруженной модели:  {loaded_pr:.4f}")

AUPRC оригинальной модели: 0.7826
AUPRC загруженной модели:  0.7826


## Сравнение с sklearn

In [57]:
pdf = df.sample(fraction=0.2, seed=42).toPandas()

In [58]:
X = pdf.drop("Class", axis=1)
y = pdf["Class"]

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [60]:
lr_sklearn = LogisticRegression(class_weight='balanced', max_iter=200, random_state=42)

pipeline_sklearn_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', lr_sklearn)
])

In [61]:
pipeline_sklearn_lr.fit(X_train, y_train)
y_pred_lr = pipeline_sklearn_lr.predict(X_test)
y_prob_lr = pipeline_sklearn_lr.predict_proba(X_test)[:, 1]

In [64]:
print(f"AUROC: {roc_auc_score(y_test, y_prob_lr):.4f}")
print(f"AUPRC: {average_precision_score(y_test, y_prob_lr):.4f}")
print(classification_report(y_test, y_pred_lr))

AUROC: 0.9547
AUPRC: 0.5054
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     11391
           1       0.05      0.87      0.10        15

    accuracy                           0.98     11406
   macro avg       0.53      0.92      0.54     11406
weighted avg       1.00      0.98      0.99     11406



In [66]:
rf_sklearn = RandomForestClassifier(
    class_weight='balanced', 
    n_estimators=50, 
    random_state=42, 
    n_jobs=-1
)

pipeline_sklearn_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', rf_sklearn)
])

In [67]:
pipeline_sklearn_rf.fit(X_train, y_train)
y_pred_rf = pipeline_sklearn_rf.predict(X_test)
y_prob_rf = pipeline_sklearn_rf.predict_proba(X_test)[:, 1]

In [68]:
print(f"AUROC: {roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"AUPRC: {average_precision_score(y_test, y_prob_rf):.4f}")
print(classification_report(y_test, y_pred_rf))

AUROC: 0.8990
AUPRC: 0.7213
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     11391
           1       0.78      0.47      0.58        15

    accuracy                           1.00     11406
   macro avg       0.89      0.73      0.79     11406
weighted avg       1.00      1.00      1.00     11406



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50926)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =